In [ ]:
from pprint import pprint

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from joblib import dump
from torch.utils.data import DataLoader, TensorDataset

# Define the neural network architecture
class Net(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        output = self.fc2(x)
        return output

# Define the training loop
def train(model, train_loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)

# Define the validation loop
def validate(model, val_loader, criterion):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for i, data in enumerate(val_loader, 0):
            inputs, labels = data
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
    return running_loss / len(val_loader)

# Define the main function
def main():
    # Define the hyperparameters
    input_size = 6
    hidden_size = 50
    output_size = 5
    learning_rate = 0.001
    num_epochs = 100
    batch_size = 32

    # Load the dataset
    df = pd.read_csv('df.csv')
    
    X = df[['datalen_bytes', 'pub_count', 'sub_count', 'best_effort', 'multicast', 'durability']].astype(float)
    X = torch.tensor(X.values, dtype=torch.float32)
    y = df[['mean_latency_us', 'mean_total_throughput_mbps', 'mean_sample_rate', 'samples_received', 'samples_lost']].astype(float)
    y = torch.tensor(y.values, dtype=torch.float32)

    # Split the dataset into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Convert the data to PyTorch tensors
    train_dataset = TensorDataset(torch.Tensor(X_train), torch.Tensor(y_train))
    val_dataset = TensorDataset(torch.Tensor(X_val), torch.Tensor(y_val))

    # Create data loaders for the training and validation sets
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Create the neural network model
    model = Net(input_size, hidden_size, output_size)

    # Define the loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_losses = []
    val_losses = []

    # Train the model
    for epoch in range(num_epochs):
        train_loss = train(model, train_loader, optimizer, criterion)
        val_loss = validate(model, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        # print(f"Epoch {epoch+1}/{num_epochs}: train_loss = {train_loss:.4f}, val_loss = {val_loss:.4f}")

    # Save the model
    torch.save(model.state_dict(), "nn_model.pth")
    
    # Plot the train_loss and val_loss
    plt.figure(figsize=(15, 10))
    plt.plot(train_losses, label='train loss')
    plt.plot(val_losses, label='val loss')
    plt.legend()
    plt.show()
    
main()